# 01 · Veri hazırlama — DisasterM3 Türkiye (Makale 1, ana veri)

**Akıştaki yeri:** `01 veri` → 02 görsel model → 03 VLM/LLM → 04 RAG → 05 değerlendirme

DisasterM3 (Wang ve ark., NeurIPS 2025; DOI 10.48550/arXiv.2505.21089; HF: `Kingdrone-Junjue/DisasterM3`, CC BY-NC-SA 4.0) deposundan **yalnız Türkiye** dosyalarını çeker (~5 GB), karo listesi (manifest) ve etiket-görüntü oturma ölçümünü üretir.

Bu not defteri veri **hazırlamaz**, doğrular: etiket ve görüntü hazır geliyor. Çıktılar:
- `manifest_dm3.csv`: karo, bölge (event), bölme (train/bench), sınıf pikselleri ve bina sayıları, öncesi/sonrası/SAR var mı.
- `hiza_dm3.csv`: her karoda etiket konturunun öncesi ve sonrası görüntüye oturması (kontur gradyan skoru) ve sonrası için FFT ile kestirilen kayma.
- Yedek veri (BRIGHT+Maxar): `01_veri_hazirlama_bright_maxar_yedek.ipynb`.

Notlar: (1) Etiket dosyası yalnız o sınıf varsa vardır; eksik maske = o sınıf yok. (2) `grid` adlı (`event_i_j`) bina maskelerinin eşleşen öncesi/sonrası görüntüsü yok; kullanılabilir karo = `status == ok`. (3) Event 0,1,3,4,5 etiketleri BRIGHT'ınkiyle piksel piksel aynı; 6 ve 7 eşleşmedi.

In [ ]:
# Ortam
%pip install -q remotezip pillow numpy pandas scipy
import os, sys
SP = os.environ.get('DM3_KOK', os.path.expanduser('~/scratch_dm3'))   # veri bu klasörün altına iner (~5 GB)
os.makedirs(SP + '/scripts', exist_ok=True)
print('Çalışma klasörü:', SP)

## 1. Türkiye dosyalarını indir
Benchmark kısmı doğrudan, eğitim kısmı zip içinden yalnız `turkey*` üyeleri (uzaktan okuma). Yeniden başlatılabilir; var olan dosyalar atlanır. Gerekli girdiler: `dm3_tree.json` (HF API `/api/datasets/Kingdrone-Junjue/DisasterM3?blobs=true` çıktısı), `train_turkey_images.json`, `train_turkey_masks.json` (zip üye listeleri; ikisi `RemoteZip(...).infolist()` ile alınır).

In [ ]:
open(SP + '/scripts/indir.py', 'w', encoding='utf8').write(r'''import json, os, sys, time, threading, urllib.request
from concurrent.futures import ThreadPoolExecutor
from remotezip import RemoteZip
SP=sys.argv[1]; OUT=os.path.join(SP,'disasterm3_turkiye'); S=os.path.join(SP,'scripts')
H="https://huggingface.co/datasets/Kingdrone-Junjue/DisasterM3/resolve/main/"
done=[0]; lk=threading.Lock()
def log(m):
    with open(os.path.join(SP,'indir.log'),'a') as f: f.write(m+'\n')
def retry(f,n=5):
    for t in range(n):
        try: return f()
        except Exception as e: err=e; time.sleep(3+4*t)
    raise err
# 1) benchmark: dosya listesindeki tüm turkey dosyaları
tree=json.load(open(os.path.join(S,'dm3_tree.json')))['siblings']
bench=[x['rfilename'] for x in tree if 'turkey' in x['rfilename'].lower() and x['rfilename'].startswith('DisasterM3_Bench/')]
def get(rel):
    p=os.path.join(OUT,'bench',rel[len('DisasterM3_Bench/'):])
    if os.path.exists(p) and os.path.getsize(p)>0: return
    os.makedirs(os.path.dirname(p),exist_ok=True)
    retry(lambda: urllib.request.urlretrieve(H+rel+"",p+'.part'))
    os.replace(p+'.part',p)
    with lk: done[0]+=1
log(f'bench dosya: {len(bench)}')
with ThreadPoolExecutor(8) as ex: list(ex.map(get,bench))
log('bench bitti')
# 2) eğitim: zip içinden yalnızca turkey
tl=threading.local()
def zget(args):
    zname,member=args
    p=os.path.join(OUT,'train',member)
    if os.path.exists(p) and os.path.getsize(p)>0: return
    if not hasattr(tl,zname): setattr(tl,zname,retry(lambda: RemoteZip(H+'DisasterM3_Instruct/'+zname)))
    rz=getattr(tl,zname)
    retry(lambda: rz.extract(member,os.path.join(OUT,'train')))
    with lk: done[0]+=1
jobs=[('masks.zip',m) for m in json.load(open(os.path.join(S,'train_turkey_masks.json')))]
jobs+=[('train_images.zip',m) for m in json.load(open(os.path.join(S,'train_turkey_images.json')))]
log(f'egitim dosya: {len(jobs)}')
with ThreadPoolExecutor(6) as ex: list(ex.map(zget,jobs))
log('egitim bitti'); log('TAMAM')
''')
print('yazıldı: {m.group(1)}')

In [ ]:
# İndirme betiğinin girdi listeleri: HF dosya ağacı + zip üye listeleri (yalnız dizinler okunur, veri inmez; ~1 dk)
import json, urllib.request
from remotezip import RemoteZip
H = 'https://huggingface.co/datasets/Kingdrone-Junjue/DisasterM3/'
tree_p = SP + '/scripts/dm3_tree.json'
if not os.path.exists(tree_p):
    open(tree_p, 'wb').write(urllib.request.urlopen(H.replace('/datasets/', '/api/datasets/').rstrip('/') + '?blobs=true', timeout=120).read())
for ad, zip_ad in [('train_turkey_masks.json', 'masks.zip'), ('train_turkey_images.json', 'train_images.zip')]:
    p = SP + '/scripts/' + ad
    if not os.path.exists(p):
        rz = RemoteZip(H + 'resolve/main/DisasterM3_Instruct/' + zip_ad)
        json.dump([i.filename for i in rz.infolist() if 'turkey' in i.filename.lower()], open(p, 'w'))
print({f: os.path.exists(SP + '/scripts/' + f) for f in ['dm3_tree.json', 'train_turkey_masks.json', 'train_turkey_images.json']})

In [ ]:
# ~5 GB iner (disk boş alanını kontrol edin). Arka planda başlar; ilerleme: {SP}/indir.log, bitince 'TAMAM' yazar.
import subprocess, shutil
print('boş alan (GB):', round(shutil.disk_usage(SP).free / 1e9, 1))
subprocess.Popen(f'cd {SP} && nohup python3 scripts/indir.py {SP} > indir.out 2>&1 &', shell=True)

## 2. Karo listesi (manifest)

In [ ]:
open(SP + '/scripts/manifest.py', 'w', encoding='utf8').write(r'''import os, re, sys, csv
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
ROOT=sys.argv[1]; OUTCSV=sys.argv[2]
CLS=['intact','damaged','destroyed']
def scan(split):
    base=os.path.join(ROOT,split)
    mdir=os.path.join(base,'masks'); pref='train' if split=='train' else 'test'
    idir=os.path.join(base,'train_images' if split=='train' else 'test_images')
    T={}   # (event, suffix) -> dict
    def add(ev,suf,k,v): T.setdefault((int(ev),suf),{})[k]=v
    for c in CLS:
        d=os.path.join(mdir,f'{pref}_building_{c}_mask')
        if os.path.isdir(d):
            for f in os.listdir(d):
                m=re.match(r'turkey_earthquake(\d+)_(.+)\.png$',f)
                if m: add(m.group(1),m.group(2),'m_'+c,os.path.join(d,f))
    if os.path.isdir(idir):
        for f in os.listdir(idir):
            m=re.match(r'turkey_earthquake(\d+)_(pre|post|sar_post)_(.+)\.png$',f)
            if m: add(m.group(1),m.group(3),m.group(2),os.path.join(idir,f))
    rows=[]
    for (ev,suf),p in sorted(T.items(),key=lambda x:(x[0][0],x[0][1])):
        r=dict(split=split,event=ev,tile=f'turkey_earthquake{ev}_{suf}',grid=bool(re.match(r'\d+_\d+$',suf)),
               has_pre='pre' in p,has_post='post' in p,has_sar='sar_post' in p,
               mask_dosya=''.join(c[0] for c in CLS if 'm_'+c in p))
        shape=None
        for c in CLS:
            if 'm_'+c in p:
                m=np.array(Image.open(p['m_'+c]))>0; shape=m.shape
                r['px_'+c]=int(m.sum()); r['n_'+c]=int(ndi.label(m)[1])
            else: r['px_'+c]=0; r['n_'+c]=0
        r['h'],r['w']=shape if shape else ('','')
        r['status']='ok' if (r['has_pre'] and r['has_post'] and shape) else ('maskesiz' if not shape else 'optik_eksik')
        rows.append(r)
    return rows
rows=[]
for s in ['train','bench']:
    if os.path.isdir(os.path.join(ROOT,s)): rows+=scan(s)
F=['split','event','tile','grid','has_pre','has_post','has_sar','mask_dosya','px_intact','px_damaged','px_destroyed','n_intact','n_damaged','n_destroyed','h','w','status']
with open(OUTCSV,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=F,extrasaction='ignore'); w.writeheader(); w.writerows(rows)
print(len(rows),'satır')
''')
print('yazıldı: {m.group(1)}')

In [ ]:
!python3 {SP}/scripts/manifest.py {SP}/disasterm3_turkiye {SP}/manifest_dm3.csv

## 3. Oturma / kayma ölçümü
Her karoda: etiket kenarındaki ortalama gradyan ÷ görüntünün ortalama gradyanı (1 = rastgele; yüksek = oturuyor), öncesi ve sonrası için; ayrıca ±64 piksel aralığında FFT ile kayma. Bakılacak: `s_pre` ile `s_post` farkı ve `dx_post, dy_post`.

In [ ]:
open(SP + '/scripts/hiza.py', 'w', encoding='utf8').write(r'''import os,sys,numpy as np,pandas as pd
from PIL import Image, ImageFilter
from concurrent.futures import ProcessPoolExecutor
ROOT=sys.argv[1]; MAN=sys.argv[2]; OUT=sys.argv[3]; R=64
def grad(a):
    g=np.array(Image.fromarray(a).convert('L').filter(ImageFilter.GaussianBlur(1.2)),dtype=np.float32)
    gy,gx=np.gradient(g); return np.hypot(gx,gy)
def edge(m): return m&~(np.roll(m,1,0)&np.roll(m,-1,0)&np.roll(m,1,1)&np.roll(m,-1,1))
def shift(G,E):
    Eb=np.array(Image.fromarray((E*255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(1.5)),dtype=np.float32)
    C=np.fft.ifft2(np.fft.fft2(G-G.mean())*np.conj(np.fft.fft2(Eb-Eb.mean()))).real
    C=np.roll(np.roll(C,R,0),R,1)[:2*R+1,:2*R+1]; i=np.unravel_index(C.argmax(),C.shape); return int(i[1]-R),int(i[0]-R)
def score(G,E,dx=0,dy=0):
    ys,xs=np.nonzero(E); return float(G[np.clip(ys+dy,0,1023),np.clip(xs+dx,0,1023)].mean()/G.mean())
def work(r):
    sp,ev,t=r
    b=os.path.join(ROOT,'train' if sp=='train' else 'bench'); md=os.path.join(b,'masks'); pf='train' if sp=='train' else 'test'
    idr=os.path.join(b,'train_images' if sp=='train' else 'test_images'); suf=t.split('_',2)[2]
    M=np.zeros((1024,1024),bool)
    for c in ['intact','damaged','destroyed']:
        p=os.path.join(md,f'{pf}_building_{c}_mask',f'{t}.png')
        if os.path.exists(p): M|=np.array(Image.open(p))>0
    E=edge(M)
    if E.sum()<500: return dict(split=sp,tile=t,n_edge=int(E.sum()))
    pre=np.array(Image.open(os.path.join(idr,f'turkey_earthquake{ev}_pre_{suf}.png')).convert('RGB'))
    po=np.array(Image.open(os.path.join(idr,f'turkey_earthquake{ev}_post_{suf}.png')).convert('RGB'))
    Gp,Go=grad(pre),grad(po)
    dxp,dyp=shift(Gp,E); dxo,dyo=shift(Go,E)
    return dict(split=sp,tile=t,n_edge=int(E.sum()),s_pre=score(Gp,E),s_post=score(Go,E),s_post_shift=score(Go,E,dxo,dyo),
                dx_pre=dxp,dy_pre=dyp,dx_post=dxo,dy_post=dyo,post_blur=float(Go.mean()/max(Gp.mean(),1e-6)),pre_mean=float(pre.mean()),post_mean=float(po.mean()))
if __name__=='__main__':
    d=pd.read_csv(MAN); d=d[d.status=='ok']
    jobs=list(zip(d.split,d.event,d.tile))
    with ProcessPoolExecutor(os.cpu_count()) as ex: res=list(ex.map(work,jobs,chunksize=8))
    pd.DataFrame(res).merge(d[['tile','split','event']],on=['tile','split']).to_csv(OUT,index=False); print(len(res))
''')
print('yazıldı: {m.group(1)}')

In [ ]:
!python3 {SP}/scripts/hiza.py {SP}/disasterm3_turkiye {SP}/manifest_dm3.csv {SP}/hiza_dm3.csv

## 4. Özet

In [ ]:
import pandas as pd, numpy as np
d = pd.read_csv(SP + '/manifest_dm3.csv'); h = pd.read_csv(SP + '/hiza_dm3.csv')
k = d[d.status == 'ok'].merge(h.drop(columns=['event', 'n_edge'], errors='ignore'), on=['split', 'tile'], how='left')
k['kayma_post_px'] = np.hypot(k.dx_post, k.dy_post)
print(k.groupby(['event']).agg(karo=('tile', 'count'), yikik_karo=('n_destroyed', lambda s: (s > 0).sum()),
      s_pre=('s_pre', 'median'), s_post=('s_post', 'median'), kayma_post_px=('kayma_post_px', 'median')).round(2))

## 5. Sonraki adımlar
- BRIGHT eşlemesi: bina pikseli üçlüsü (sağlam, hasarlı, yıkık) BRIGHT karolarıyla eşleşince event → şehir ve koordinat bilgisi elde edilir (0 Gaziantep, 1 Adıyaman, 3 Malatya, 4 Antakya/Defne, 5 Osmaniye).
- Event 5 ve diğerlerinde sonrası görüntü kayması: yedek not defterindeki hizalama (5c) aynı yöntemle uygulanabilir.
- Denetim örneklemesi (tabakalı ~100 karo) ve bölme kararı: `Notlar/` altındaki karar notuna bakın.